# Semana 10: aumento em tempo de teste (TTA), margens das 15 execuções

Análise secundária da seção 9 de `prereg/preregistro.md`, rodada depois da abertura da chave dos regimes (`prereg/desvios.md`, entradas de 25/10/2026 e de 05/11/2026). Para cada uma das 15 execuções confirmatórias (R1, R2 e R3, sementes 1 a 5), calcula a margem com as 8 transformações diedrais no teste da fonte, na Bolívia e nas 561 janelas do T2. A definição do TTA, a montagem do modelo, o pré-processamento, a inferência em 32 bits e o lote de 8 são os do notebook da semana 6. **Os arquivos da semana 6 não são alterados**: a saída vai para uma pasta nova. **Este notebook não calcula nenhuma métrica.**

| Célula | O que faz |
|---|---|
| 1 | Instala o TerraTorch 1.2.11 |
| 2 | Monta o Drive, clona o repositório, confere a chave dos códigos, os pesos e o manifesto da semana 6 |
| 3 | Confere o backbone |
| 4 | Baixa o teste da fonte e a Bolívia e confere os rótulos com os da semana 6 |
| 5 | Lê as 561 janelas do T2 e confere os rótulos com os da semana 6 |
| 6 | Monta o modelo (código da semana 6) |
| 7 | Roda o TTA nas 15 execuções. Em cada conjunto, a margem sem transformação sai da mesma passada e é comparada com a gravada na semana 6; acima de 1e-3, nada é gravado e a célula para |
| 8 | Remonta o Drive, confere os arquivos pelo SHA-256 e grava o manifesto |

**Saída** em `resultados/semana10/tta_margens`: `tta_<código>_s<semente>_<conjunto>.npz` (`chaves`, `tamanhos` e `margem_tta` em float32, um valor por pixel pontuado, na ordem dos `labels_<conjunto>.npz` da semana 6), `conferencia_<código>_s<semente>.json`, `manifesto_sha256.csv` e `info_tta.json`.

Abra o Colab logada na conta do mestrado, **com GPU T4**: as margens da semana 6 foram feitas numa T4, e a comparação da célula 7 espera diferença zero. O TTA leva cerca de 8 vezes o tempo da inferência simples da semana 6, então conte com várias horas. **Se a sessão cair**, rode de novo a partir da célula 1 (ou da 2, se o TerraTorch ainda estiver instalado); o que já foi gravado é pulado. Se alguma célula der erro, pare e guarde a saída inteira.

Depois da célula 8, me mande `manifesto_sha256.csv` e `info_tta.json`. A análise roda depois, em outro notebook, sem GPU.

In [ ]:
# Célula 1 de 8: instalação
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"
!pip uninstall -y -q torchao

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

import sys
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != version('numpy'):
    print(f"\nREINICIE A SESSÃO: o NumPy carregado é o {sys.modules['numpy'].__version__} e o instalado é o {version('numpy')}.")
    print('Menu Ambiente de execução > Reiniciar sessão (não precisa excluir o ambiente) e rode a partir da célula 2.')
else:
    print('\nNumPy carregado e instalado coincidem. Pode seguir para a célula 2.')

In [ ]:
# Célula 2 de 8: Drive, repositório, chave, pesos e manifesto da semana 6
import os, re, json, time, hashlib, shutil, glob, gc, csv, subprocess, sys
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

S5 = f'{RAIZ}/resultados/semana5'
S2R = f'{RAIZ}/resultados/semana2'
MARGENS6 = f'{RAIZ}/resultados/semana6/margens'
CHAVE6 = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
HASHES_S1F11 = f'{RAIZ}/resultados/semana6/entradas_sen1floods11_sha256.csv'
JANELAS = f'{RAIZ}/dados/worldfloods_v2/janelas'
SAIDA = f'{RAIZ}/resultados/semana10/tta_margens'      # pasta fixa: uma sessão nova continua de onde a anterior parou
LOCAL = '/content/tta'
assert not os.path.exists(f'{SAIDA}/manifesto_sha256.csv'), (
    'A pasta de saída já tem uma rodada completa (manifesto_sha256.csv). Nada é gravado por cima dela: '
    'para uma rodada nova, mude SAIDA para resultados/semana10/tta_margens_2.')
for d in (SAIDA, LOCAL):
    os.makedirs(d, exist_ok=True)
for resto in glob.glob(f'{SAIDA}/parcial_*'):
    os.remove(resto)                                    # cópia interrompida de uma sessão anterior

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def esperar(caminho, limite=180):
    t0 = time.time()
    while not os.path.exists(caminho):
        if time.time() - t0 > limite:
            raise FileNotFoundError(f'{caminho} não apareceu no Drive em {limite} s. Pare e guarde esta saída.')
        time.sleep(5)

def trazer(no_drive, sha_esperado, pasta=LOCAL):
    # Copia do Drive para o disco local e confere o SHA-256; se não conferir, remonta o Drive uma vez e copia de novo.
    local = f'{pasta}/{os.path.basename(no_drive)}'
    for tentativa in (1, 2):
        esperar(no_drive)
        if not (os.path.exists(local) and sha256_arquivo(local) == sha_esperado):
            shutil.copyfile(no_drive, local + '.parcial')
            os.replace(local + '.parcial', local)
        sha = sha256_arquivo(local)
        if sha == sha_esperado:
            return local
        if tentativa == 1:
            print(f'   {os.path.basename(no_drive)}: SHA-256 diferente do registro; remontando o Drive e copiando de novo.')
            drive.flush_and_unmount()
            time.sleep(10)
            drive.mount('/content/drive', force_remount=True)
    raise ValueError(f'{no_drive}: SHA-256 {sha[:16]}..., o registro diz {sha_esperado[:16]}... Pare e guarde esta saída.')

def gravar_no_drive(local, nome):
    # Copia um arquivo pronto do disco local para a SAIDA com nome provisório e só então dá o nome final.
    parcial = f'{SAIDA}/parcial_{nome}'
    shutil.copyfile(local, parcial)
    os.replace(parcial, f'{SAIDA}/{nome}')

def por_de_lado(nome):
    # Arquivo de uma gravação interrompida: fica no Drive com outro nome, nada é apagado.
    marca = time.strftime('%Y%m%dT%H%M%S')
    for sufixo in ('', '.sha256'):
        if os.path.exists(f'{SAIDA}/{nome}{sufixo}'):
            os.replace(f'{SAIDA}/{nome}{sufixo}', f'{SAIDA}/descartado_{marca}_{nome}{sufixo}')

def carregar_npz(caminho):
    with np.load(caminho, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()

# Chave dos códigos da semana 6, aberta na rodada confirmatória (prereg/desvios.md)
CHAVE = json.load(open(CHAVE6))
assert CHAVE == {'8tjg': 'R1', '4m9f': 'R2', '6wke': 'R3'}, 'A chave da semana 6 não é a registrada em prereg/desvios.md.'
CODIGO = {r: c for c, r in CHAVE.items()}

REGIMES = ('R1', 'R2', 'R3')
SEMENTES = (1, 2, 3, 4, 5)
CONJUNTOS = ('teste', 'bolivia', 't2')
PESOS = {}
for regime in REGIMES:
    for semente in SEMENTES:
        boas = []
        for tentativa in (1, 2):
            registro = f'{S5}/exec_{regime}_s{semente}_t{tentativa}.json'
            if os.path.exists(registro) and json.load(open(registro)).get('situacao') == 'ok':
                boas.append(registro)
        pesos = f'{S5}/pesos_{regime}_s{semente}.pt'
        assert len(boas) == 1 and os.path.exists(pesos), f'Faltam arquivos do treino {regime} s{semente} em {S5}.'
        PESOS[(regime, semente)] = dict(pesos=pesos, bytes=os.path.getsize(pesos))
print(f'{len(PESOS)} execuções de treino encontradas.')

# Manifesto das margens da semana 6: os arquivos lidos aqui são conferidos por ele
MANIFESTO6 = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(f'{MARGENS6}/manifesto_sha256.csv', newline=''))}
assert len(MANIFESTO6) == 64, f'O manifesto da semana 6 tem {len(MANIFESTO6)} arquivos, esperado 64 (60 execuções e 4 rótulos).'

livre = shutil.disk_usage(MEUDRIVE)[2]
print(f'Espaço livre no Drive: {livre / 1e9:.0f} GB. As margens do TTA ocupam cerca de 8 GB.')
assert livre > 10e9, 'Espaço livre no Drive insuficiente para as margens do TTA (mínimo de 10 GB).'
print(f'Repositório no commit {commit[:7]}. Saída em {SAIDA.replace(MEUDRIVE, "MyDrive")}')

In [ ]:
# Célula 3 de 8: backbone
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO = '63adbd39c271da4c42f447e69b1a7c91a338cdc9'
SHA_BACKBONE = '3629cedfbb350faafcb0dac902ae0d3c927e25ce8d9e0024aa1276ec66956ddb'

no_drive = f'{RAIZ}/modelo_fixado/backbone/{ARQUIVO}'
if not os.path.exists(no_drive):
    print('Backbone ausente no Drive; baixando na revisão fixada.')
    hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=os.path.dirname(no_drive))
BACKBONE_PT = f'/content/{ARQUIVO}'
if not (os.path.exists(BACKBONE_PT) and sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE):
    shutil.copy(no_drive, BACKBONE_PT)
assert sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE, 'sha256 do backbone diferente do registrado em modelo.md.'
print(f'Backbone conferido ({SHA_BACKBONE[:8]}…), cópia local em {BACKBONE_PT}')

In [ ]:
# Célula 4 de 8: teste da fonte e Bolívia, com os rótulos conferidos com os da semana 6
import urllib.request, concurrent.futures as cf
import rasterio

BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/'
DADOS = '/content/sen1floods11'
S2 = f'{DADOS}/v1.1/data/flood_events/HandLabeled/S2Hand'
LB = f'{DADOS}/v1.1/data/flood_events/HandLabeled/LabelHand'
for d in (S2, LB):
    os.makedirs(d, exist_ok=True)
TAM = 512
IDX_BANDAS = [1, 2, 3, 8, 11, 12]          # B2, B3, B4, B8A, B11, B12 nos 13 canais do Sen1Floods11
ESCALA = np.float32(0.0001)
N_ESPERADO = {'teste': 90, 'bolivia': 15, 't2': 561}

hashes = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(HASHES_S1F11, newline=''))}
ROTULOS = {}
for conj in CONJUNTOS:
    nome = f'labels_{conj}.npz'
    local = trazer(f'{MARGENS6}/{nome}', MANIFESTO6[nome])
    ROTULOS[conj] = carregar_npz(local)
    os.remove(local)
    assert len(ROTULOS[conj]['chaves']) == N_ESPERADO[conj], f'{nome}: {len(ROTULOS[conj]["chaves"])} unidades.'

def baixar(trabalho):
    url, destino = trabalho
    nome = os.path.basename(destino)
    if not (os.path.exists(destino) and sha256_arquivo(destino) == hashes[nome]):
        for tentativa in range(4):
            try:
                urllib.request.urlretrieve(url, destino + '.parcial')
                os.replace(destino + '.parcial', destino)
                break
            except Exception:
                time.sleep(2 * (tentativa + 1))
    assert sha256_arquivo(destino) == hashes[nome], f'{nome} diferente do hash gravado na semana 6.'
    return nome

def preencher(x, valor):
    x = np.ma.filled(x.astype(np.float32), np.nan)
    return np.where(np.isnan(x), np.float32(valor), x).astype(np.float32)

def ler_chip(chave):
    with rasterio.open(f'{S2}/{chave}_S2Hand.tif') as s:
        assert s.count == 13 and (s.height, s.width) == (TAM, TAM), f'{chave}: imagem'
        x = preencher(s.read(masked=True), 0.0)[IDX_BANDAS]
    with rasterio.open(f'{LB}/{chave}_LabelHand.tif') as s:
        assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{chave}: rótulo'
        y = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    return x, y

def conferir_rotulos(conj, ys):
    r = ROTULOS[conj]
    pont = ys >= 0
    assert np.array_equal(pont.reshape(len(ys), -1).sum(1), r['tamanhos']), f'{conj}: pixels pontuados diferentes dos da semana 6.'
    assert np.array_equal(ys[pont].astype(np.uint8), r['rotulo']), f'{conj}: rótulos diferentes dos da semana 6.'

CONJ = {}
for conj in ('teste', 'bolivia'):
    chaves = [str(c) for c in ROTULOS[conj]['chaves']]
    trabalhos = []
    for chave in chaves:
        trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
        trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
    with cf.ThreadPoolExecutor(16) as pool:
        list(pool.map(baixar, trabalhos))
    X = np.empty((len(chaves), len(IDX_BANDAS), TAM, TAM), np.float32)
    Y = np.empty((len(chaves), TAM, TAM), np.int16)
    for i, c in enumerate(chaves):
        X[i], Y[i] = ler_chip(c)
    conferir_rotulos(conj, Y)
    CONJ[conj] = dict(chaves=chaves, bruto=X, ys=Y)
    print(f'{conj}: {len(chaves)} chips conferidos pelos hashes da semana 6, {int(ROTULOS[conj]["tamanhos"].sum()):,} pixels pontuados; '
          'rótulos iguais aos da semana 6.')

In [ ]:
# Célula 5 de 8: as 561 janelas do T2, com os rótulos conferidos com os da semana 6
def ler_tabela(nome, sha_esperado):
    local = trazer(f'{S2R}/{nome}', sha_esperado)
    with open(local, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

TAB_JANELAS = ler_tabela('t2_janelas.csv', '3c21970835f2d43cd436be9616c451ebbff5fcea138b7401d3867f0901643b2c')
TAB_NUVEM = ler_tabela('t2_janelas_nuvem050.csv', '3b46e703738042b235b68a12600cfc0c7bf917b3cf5dec7efd8b924a02e132a6')
entram = {r['window_id'] for r in TAB_JANELAS + TAB_NUVEM if r['kept_sensitivity'] == 'True'}
chaves = [str(c) for c in ROTULOS['t2']['chaves']]
assert set(chaves) == entram and len(chaves) == len(entram), 'As janelas de labels_t2.npz não são as 561 das tabelas de janelas.'

X = np.lib.format.open_memmap(f'{LOCAL}/t2.npy', mode='w+', dtype=np.int16, shape=(len(chaves), len(IDX_BANDAS), TAM, TAM))
Y = np.empty((len(chaves), TAM, TAM), np.int16)
t0 = time.time()
for i, jid in enumerate(chaves):
    with rasterio.open(f'{JANELAS}/S2/{jid}.tif') as s:
        assert s.count == 6 and (s.height, s.width) == (TAM, TAM) and s.dtypes[0] == 'int16', f'{jid}: imagem'
        X[i] = s.read()
    with rasterio.open(f'{JANELAS}/label/{jid}.tif') as s:
        assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{jid}: rótulo'
        Y[i] = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    if (i + 1) % 100 == 0:
        print(f'   {i + 1} de {len(chaves)} janelas lidas ({time.time() - t0:.0f} s)')
X.flush()
conferir_rotulos('t2', Y)
CONJ['t2'] = dict(chaves=chaves, bruto=X, ys=Y)
print(f't2: {len(chaves)} janelas lidas em {time.time() - t0:.0f} s, {int(ROTULOS["t2"]["tamanhos"].sum()):,} pixels pontuados; '
      'rótulos iguais aos da semana 6.')

In [ ]:
# Célula 6 de 8: modelo (código da semana 6)
import warnings, contextlib, io, logging, gc
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from importlib.metadata import version
from scipy.special import log_expit, logsumexp
try:
    from terratorch.tasks import SemanticSegmentationTask
    from terratorch.datamodules import Sen1Floods11NonGeoDataModule
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 e confira que ela imprime terratorch 1.2.11.') from erro

warnings.filterwarnings('ignore', category=UserWarning)
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4.'
DISPOSITIVO = 'cuda'
BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
ESPERADO_M = {'R1': 15.1, 'R2': 20.6, 'R3': 319.0}
TOLERANCIA_M = 0.1
LR = {'R1': 5e-4, 'R2': 5e-4, 'R3': 5e-5}
EPOCAS = 50
LOTE_INF = 8
LIMITE_DIF_VAL = 0.05

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def datamodule():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE_INF, num_workers=0, bands=BANDAS,
        train_transform=[para_tensor()], val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=False, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

LORA = dict(method='LORA', replace_qkv='qkv',
            peft_config_kwargs=dict(target_modules=['qkv.q_linear', 'qkv.v_linear', 'mlp.fc1', 'mlp.fc2'],
                                    lora_alpha=16, r=16))

def tarefa(regime):
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True)
    if regime == 'R2':
        args['peft_config'] = LORA
    t = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=LR[regime], optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        scheduler='CosineAnnealingLR', scheduler_hparams={'T_max': EPOCAS, 'eta_min': 0},
        freeze_backbone=(regime == 'R1'), plot_on_val=0)
    t.configure_models()
    treinaveis = sum(p.numel() for p in t.parameters() if p.requires_grad)
    if regime == 'R2':
        n_lora = sum(1 for n, _ in t.model.encoder.named_parameters() if 'lora_A' in n)
        assert n_lora == 96, 'Número de módulos LoRA diferente do esperado.'
    assert abs(treinaveis / 1e6 - ESPERADO_M[regime]) <= TOLERANCIA_M, 'Parâmetros treináveis fora do esperado.'
    return t

DM = datamodule()

@contextlib.contextmanager
def em_silencio():
    """Esconde o que a construção do modelo imprime, que diferiria entre os regimes."""
    logging.disable(logging.CRITICAL)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)

def carregar_modelo(regime, semente):
    with em_silencio():
        t = tarefa(regime)
        t.model.to(DISPOSITIVO)
        gc.collect()
        pesos = torch.load(PESOS[(regime, semente)]['pesos'], map_location=DISPOSITIVO)
        do_modelo = set(t.model.state_dict().keys())
        esperado = do_modelo if regime != 'R1' else {k for k in do_modelo if not k.startswith('encoder.')}
        assert set(pesos.keys()) == esperado, 'Os tensores salvos não coincidem com os do modelo.'
        resultado = t.model.load_state_dict(pesos, strict=False)
        assert not resultado.unexpected_keys, 'Tensores salvos sem lugar no modelo.'
        assert set(resultado.missing_keys) == (do_modelo - esperado), 'Tensores do modelo sem peso.'
        modelo = t.model.to(DISPOSITIVO).float().eval()
        for p in modelo.parameters():
            p.requires_grad_(False)
        del t, pesos
    gc.collect()
    return modelo

TRANSFORMACOES = [(k, espelho) for espelho in (False, True) for k in range(4)]

def aplicar(x, k, espelho):
    if espelho:
        x = torch.flip(x, dims=(-1,))
    return torch.rot90(x, k, dims=(-2, -1))

def desfazer(y, k, espelho):
    y = torch.rot90(y, -k, dims=(-2, -1))
    if espelho:
        y = torch.flip(y, dims=(-1,))
    return y

def margem_media_probabilidade(pilha):
    """Logit da probabilidade média de água de K margens, pilha (K, N), em float64 (igual a nucleo.mean_probability_margin)."""
    pilha = np.asarray(pilha, dtype=np.float64)
    k = pilha.shape[0]
    lp = logsumexp(log_expit(pilha), axis=0) - np.log(k)
    lq = logsumexp(log_expit(-pilha), axis=0) - np.log(k)
    return lp - lq

def margens_tta(modelo, conj):
    """Margem sem transformação e margem do TTA (logit da probabilidade média de água sobre as 8 transformações
    do grupo diedral) de cada pixel pontuado, em 32 bits, chips inteiros, como inferir(..., tta=True) da semana 6.
    A margem sem transformação é a da primeira transformação (rotação 0, sem espelho), da mesma passada."""
    assert TRANSFORMACOES[0] == (0, False)
    bruto, ys = conj['bruto'], conj['ys']
    simples, tta = [], []
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        for i in range(0, len(ys), LOTE_INF):
            x = torch.from_numpy(bruto[i:i + LOTE_INF].astype(np.float32) * ESCALA).to(DISPOSITIVO)
            y = torch.from_numpy(ys[i:i + LOTE_INF].astype(np.int64)).to(DISPOSITIVO)
            x = DM.aug({'image': x, 'mask': y})['image']
            assert x.dtype == torch.float32 and x.ndim == 4 and x.shape[1] == len(BANDAS), tuple(x.shape)
            pilhas = []
            for k, espelho in TRANSFORMACOES:
                z = modelo(aplicar(x, k, espelho)).output.float()
                assert tuple(z.shape[-2:]) == tuple(x.shape[-2:]), f'Saída {tuple(z.shape)}'
                pilhas.append(desfazer(z[:, 1] - z[:, 0], k, espelho).cpu().numpy())
            for j in range(x.shape[0]):
                pont = ys[i + j] >= 0
                simples.append(pilhas[0][j][pont].astype(np.float32))
                tta.append(margem_media_probabilidade(np.stack([p[j][pont] for p in pilhas])).astype(np.float32))
    return np.concatenate(simples), np.concatenate(tta)

print('Definições carregadas. Nada foi inferido ainda.')

In [ ]:
# Célula 7 de 8: TTA nas 15 execuções
import psutil
LIMITE_RAM_PCT = 62
LIMITE_SEM_TRANSFORMACAO = 1e-3

def nome_tta(cod, semente, conj):
    return f'tta_{cod}_s{semente}_{conj}.npz'

def feito(cod, semente, conj):
    nome = nome_tta(cod, semente, conj)
    return os.path.exists(f'{SAIDA}/{nome}') and os.path.exists(f'{SAIDA}/{nome}.sha256')

for regime in REGIMES:
    for semente in SEMENTES:
        cod = CODIGO[regime]
        faltam = [c for c in CONJUNTOS if not feito(cod, semente, c)]
        if not faltam:
            print(f'{regime} s{semente}: já feita.')
            continue
        pct = psutil.virtual_memory().percent
        if pct > LIMITE_RAM_PCT:
            raise RuntimeError(f'PAREI: RAM em {pct:.0f}%. Reinicie a sessão e rode da célula 2 em diante; o que está feito é pulado.')
        t0 = time.time()
        modelo = carregar_modelo(regime, semente)
        arq_conf = f'{SAIDA}/conferencia_{cod}_s{semente}.json'
        conf = (json.load(open(arq_conf)) if os.path.exists(arq_conf)
                else {'regime': regime, 'semente': semente, 'codigo': cod, 'conjuntos': {}})
        for conj in faltam:
            t1 = time.time()
            with em_silencio():
                simples, tta = margens_tta(modelo, CONJ[conj])
            r = ROTULOS[conj]
            n_pix = int(r['tamanhos'].sum())
            assert simples.dtype == np.float32 and tta.dtype == np.float32 and simples.size == tta.size == n_pix, (
                f'{regime} s{semente} {conj}: margens com tamanho errado; nada foi gravado.')
            assert np.isfinite(tta).all(), f'{regime} s{semente} {conj}: margens do TTA não finitas; nada foi gravado.'

            # A margem sem transformação tem de ser a gravada na semana 6
            nome6 = f'run_{cod}_s{semente}_{conj}.npz'
            local6 = trazer(f'{MARGENS6}/{nome6}', MANIFESTO6[nome6])
            z6 = carregar_npz(local6)
            os.remove(local6)
            assert [str(c) for c in z6['chaves']] == [str(c) for c in r['chaves']] and np.array_equal(z6['tamanhos'], r['tamanhos'])
            dif = float(np.max(np.abs(simples - z6['margem']))) if n_pix else 0.0
            trocas = int(np.sum((simples > 0) != (z6['margem'] > 0)))
            assert dif <= LIMITE_SEM_TRANSFORMACAO, (
                f'{regime} s{semente} {conj}: a margem sem transformação difere da gravada na semana 6 em {dif:.2e} '
                f'({trocas} pixels com sinal trocado). Nada foi gravado. Pare e guarde esta saída.')

            nome = nome_tta(cod, semente, conj)
            por_de_lado(nome)
            np.savez_compressed(f'{LOCAL}/{nome}', chaves=r['chaves'], tamanhos=r['tamanhos'], margem_tta=tta)
            sha = sha256_arquivo(f'{LOCAL}/{nome}')
            gravar_no_drive(f'{LOCAL}/{nome}', nome)
            with open(f'{SAIDA}/{nome}.sha256', 'w') as f:
                f.write(sha + '\n')
            os.remove(f'{LOCAL}/{nome}')
            conf['conjuntos'][conj] = {'pixels': n_pix, 'diferenca_maxima_sem_transformacao': dif,
                                       'pixels_com_sinal_trocado': trocas, 'segundos': round(time.time() - t1)}
            conf['gpu'] = torch.cuda.get_device_name(0)
            json.dump(conf, open(arq_conf, 'w'), indent=2)
            print(f'   {regime} s{semente} {conj}: diferença sem transformação {dif:.1e} ({trocas} pixels com sinal trocado), '
                  f'{time.time() - t1:.0f} s')
            del simples, tta, z6
        print(f'{regime} s{semente}: {len(faltam)} conjunto(s) gravado(s) em {(time.time() - t0) / 60:.0f} min '
              f'(RAM em {psutil.virtual_memory().percent:.0f}%).')
        del modelo
        gc.collect(); torch.cuda.empty_cache()
print('\nGuarde a saída desta célula.')

In [ ]:
# Célula 8 de 8: conferência final
from importlib.metadata import version

print('Gravando o Drive e conferindo de novo; pode levar alguns minutos.')
drive.flush_and_unmount()
time.sleep(10)
drive.mount('/content/drive', force_remount=True)

linhas, ruins = [], []
for regime in REGIMES:
    for semente in SEMENTES:
        cod = CODIGO[regime]
        for conj in CONJUNTOS:
            nome = nome_tta(cod, semente, conj)
            esperar(f'{SAIDA}/{nome}')
            assert os.path.exists(f'{SAIDA}/{nome}.sha256'), f'Falta {nome}.sha256. Rode a célula 7 de novo.'
            sha = sha256_arquivo(f'{SAIDA}/{nome}')
            if sha != open(f'{SAIDA}/{nome}.sha256').read().strip():
                ruins.append(nome)
                continue
            linhas.append([nome, os.path.getsize(f'{SAIDA}/{nome}'), sha])
for nome in ruins:
    por_de_lado(nome)
assert not ruins, (f'{len(ruins)} arquivo(s) no Drive diferente(s) do que foi gravado: {ruins}. Eles foram renomeados para '
                   f'descartado_...; rode a célula 7 de novo (só eles serão refeitos) e depois esta.')

CONF = []
for regime in REGIMES:
    for semente in SEMENTES:
        c = json.load(open(f'{SAIDA}/conferencia_{CODIGO[regime]}_s{semente}.json'))
        assert sorted(c['conjuntos']) == sorted(CONJUNTOS), f'{regime} s{semente}: conferência incompleta.'
        CONF.append(c)
dif_max = max(v['diferenca_maxima_sem_transformacao'] for c in CONF for v in c['conjuntos'].values())
trocas = sum(v['pixels_com_sinal_trocado'] for c in CONF for v in c['conjuntos'].values())

with open(f'{SAIDA}/manifesto_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['arquivo', 'bytes', 'sha256'])
    w.writerows(linhas)
INFO = {'analise': 'secundaria preregistrada (prereg/preregistro.md, secao 9), rodada apos abertura da chave',
        'desvios': 'prereg/desvios.md, entradas de 2026-10-25 e 2026-11-05',
        'commit_repositorio': commit, 'codigos_semana6': CODIGO, 'regimes': list(REGIMES), 'sementes': list(SEMENTES),
        'conjuntos': {c: {'unidades': len(ROTULOS[c]['chaves']), 'pixels': int(ROTULOS[c]['tamanhos'].sum())} for c in CONJUNTOS},
        'transformacoes': '8 do grupo diedral: rotacoes de 0, 90, 180 e 270 graus, sem e com espelho horizontal',
        'margem_tta': 'logit da probabilidade media de agua sobre as 8 transformacoes (media em float64, gravada em float32)',
        'limite_sem_transformacao': LIMITE_SEM_TRANSFORMACAO, 'diferenca_maxima_sem_transformacao': dif_max,
        'pixels_com_sinal_trocado': trocas, 'conferencias': CONF,
        'pesos': {f'{r}_s{s}': {'arquivo': os.path.basename(v['pesos']), 'bytes': v['bytes']} for (r, s), v in PESOS.items()},
        'backbone_sha256': SHA_BACKBONE, 'precisao_inferencia': 'float32', 'lote': LOTE_INF,
        'versoes': {q: version(q) for q in ('terratorch', 'torchgeo', 'peft', 'torch', 'numpy')},
        'gpu': torch.cuda.get_device_name(0), 'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA}/info_tta.json', 'w'), indent=2, ensure_ascii=False)
print(f'{len(linhas)} arquivos de margens do TTA conferidos depois de remontar o Drive.')
print(f'Margem sem transformação contra a semana 6: diferença máxima {dif_max:.1e}, {trocas} pixels com sinal trocado.')
print('Manifesto e info_tta.json gravados em resultados/semana10/tta_margens. Nenhuma métrica foi calculada.')